Zadanie 1

Cel zadania: Wykonanie różnych typów jonów i sprawdzenie jaki to ma wpływ na wyniki zapytania.  

Wykonaj połączenia w notatniku Joins.dbc 

Dla tych jonów porównaj inner i left and right outer, Left Semi Join, Left Anti Join join  

za każdym razem sprawdź .explain i zobacz jak Spark wykonuje połączenia 

In [0]:
%scala
import spark.implicits._
val person = Seq(
  (0, "Bill Chambers", 0, Seq(100)), 
  (1, "Matei Zaharia", 1, Seq(500, 250, 100)),
  (2, "Michael Armbrust", 1, Seq(250, 100))
  ).toDF("id", "name", "graduate_program", "spark_status")


val graduateProgram = Seq(
  (0, "Masters", "School of Information", "UC Berkeley"),
  (2, "Masters", "EECS", "UC Berkeley"),
  (1, "Ph.D.", "EECS", "UC Berkeley")
).toDF("id", "degree", "department", "school")

val sparkStatus = Seq(
  (500, "Vice President"),
  (250, "PMC Member"),
  (100, "Contributor")
).toDF("id", "status")


import spark.implicits._
person: org.apache.spark.sql.DataFrame = [id: int, name: string ... 2 more fields]
graduateProgram: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 2 more fields]
sparkStatus: org.apache.spark.sql.DataFrame = [id: int, status: string]

In [0]:
%scala
val joinExpression = person.col("graduate_program") === graduateProgram.col("id")
val wrongJoinExpression = person.col("name") === graduateProgram.col("school")
person.join(graduateProgram, joinExpression).show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
 id| name|graduate_program| spark_status| id| degree| department| school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
 0| Bill Chambers| 0| [100]| 0|Masters|School of Informa...|UC Berkeley|
 2|Michael Armbrust| 1| [250, 100]| 1| Ph.D.| EECS|UC Berkeley|
 1| Matei Zaharia| 1|[500, 250, 100]| 1| Ph.D.| EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+

joinExpression: org.apache.spark.sql.Column = (graduate_program = id)
wrongJoinExpression: org.apache.spark.sql.Column = (name = school)

In [0]:
%scala
person.join(graduateProgram, wrongJoinExpression).show()

+---+----+----------------+------------+---+------+----------+------+
 id|name|graduate_program|spark_status| id|degree|department|school|
+---+----+----------------+------------+---+------+----------+------+
+---+----+----------------+------------+---+------+----------+------+

In [0]:
%scala
var joinType = "inner"
person.join(graduateProgram, joinExpression, joinType).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [graduate_program#1009], [id#1028], Inner, BuildLeft, false, true
 :- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=2780]
 : +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]
 +- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]


joinType: String = inner

In [0]:
%scala
joinType = "outer"
val joinedDF = person.join(graduateProgram, joinExpression, joinType)
joinedDF.explain()
joinedDF.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#1009], [id#1028], FullOuter
 :- Sort [graduate_program#1009 ASC NULLS FIRST], false, 0
 : +- Exchange hashpartitioning(graduate_program#1009, 200), ENSURE_REQUIREMENTS, [plan_id=2800]
 : +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]
 +- Sort [id#1028 ASC NULLS FIRST], false, 0
 +- Exchange hashpartitioning(id#1028, 200), ENSURE_REQUIREMENTS, [plan_id=2801]
 +- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]


+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
 id| name|graduate_program| spark_status| id| degree| department| school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
 0| Bill Chambers| 0| [100]| 0|Masters|School of Informa...|UC Berkeley|
 1| Matei Zaharia| 1|[500, 250, 100]| 1| Ph.D.| EECS|UC Berkeley|
 2|Michael Armbrust| 1| [250, 100]| 1| Ph.D.| EECS|UC Berkeley|
null| null| null| null| 2|Masters| EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+

joinType: String = outer
joinedDF: org.apache.spark.sql.DataFrame = [id: int, name: string ... 6 more fields]

In [0]:
%scala
joinType = "left_outer"
val joinedDF2 = graduateProgram.join(person, joinExpression, joinType)
joinedDF2.explain()
joinedDF2.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], LeftOuter, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=2914]
 +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]


+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
 id| degree| department| school| id| name|graduate_program| spark_status|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+
 0|Masters|School of Informa...|UC Berkeley| 0| Bill Chambers| 0| [100]|
 2|Masters| EECS|UC Berkeley|null| null| null| null|
 1| Ph.D.| EECS|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 1| Ph.D.| EECS|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
+---+-------+--------------------+-----------+----+----------------+----------------+---------------+

joinType: String = left_outer
joinedDF2: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 6 more fields]

In [0]:
%scala
joinType = "right"
val joinedDF3 = person.join(graduateProgram, joinExpression, joinType)
joinedDF3.explain()
joinedDF3.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [graduate_program#1009], [id#1028], RightOuter, BuildLeft, false, true
 :- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=2981]
 : +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]
 +- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]


+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
 id| name|graduate_program| spark_status| id| degree| department| school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
 0| Bill Chambers| 0| [100]| 0|Masters|School of Informa...|UC Berkeley|
null| null| null| null| 2|Masters| EECS|UC Berkeley|
 2|Michael Armbrust| 1| [250, 100]| 1| Ph.D.| EECS|UC Berkeley|
 1| Matei Zaharia| 1|[500, 250, 100]| 1| Ph.D.| EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+

joinType: String = right
joinedDF3: org.apache.spark.sql.DataFrame = [id: int, name: string ... 6 more fields]

In [0]:
%scala
joinType = "left_semi"
val joinedDF4 = graduateProgram.join(person, joinExpression, joinType)
joinedDF4.explain()
joinedDF4.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], LeftSemi, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3048]
 +- LocalTableScan [graduate_program#1009]


+---+-------+--------------------+-----------+
 id| degree| department| school|
+---+-------+--------------------+-----------+
 0|Masters|School of Informa...|UC Berkeley|
 1| Ph.D.| EECS|UC Berkeley|
+---+-------+--------------------+-----------+

joinType: String = left_semi
joinedDF4: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 2 more fields]

In [0]:
%scala
val gradProgram2 = graduateProgram.union(Seq(
    (0, "Masters", "Duplicated Row", "Duplicated School")).toDF())

gradProgram2.createOrReplaceTempView("gradProgram2")
val joinedDF5 = gradProgram2.join(person, joinExpression, joinType)
joinedDF5.explain()
joinedDF5.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], LeftSemi, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3121]
 +- LocalTableScan [graduate_program#1009]


+---+-------+--------------------+-----------------+
 id| degree| department| school|
+---+-------+--------------------+-----------------+
 0|Masters|School of Informa...| UC Berkeley|
 1| Ph.D.| EECS| UC Berkeley|
 0|Masters| Duplicated Row|Duplicated School|
+---+-------+--------------------+-----------------+

gradProgram2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: int, degree: string ... 2 more fields]
joinedDF5: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 2 more fields]

In [0]:
%scala
joinType = "left_anti"
val joinedDF6 = graduateProgram.join(person, joinExpression, joinType)
joinedDF6.explain()
joinedDF6.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], LeftAnti, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3188]
 +- LocalTableScan [graduate_program#1009]


+---+-------+----------+-----------+
 id| degree|department| school|
+---+-------+----------+-----------+
 2|Masters| EECS|UC Berkeley|
+---+-------+----------+-----------+

joinType: String = left_anti
joinedDF6: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 2 more fields]

In [0]:
%scala
joinType = "cross"
val joinedDF7 = graduateProgram.join(person, joinExpression, joinType)
joinedDF7.explain()
joinedDF7.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], Cross, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3255]
 +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]


+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 id| degree| department| school| id| name|graduate_program| spark_status|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 0|Masters|School of Informa...|UC Berkeley| 0| Bill Chambers| 0| [100]|
 1| Ph.D.| EECS|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 1| Ph.D.| EECS|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+

joinType: String = cross
joinedDF7: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 6 more fields]

In [0]:
%scala
val joinedDF8 = graduateProgram.join(person, joinExpression, joinType)
joinedDF8.explain()
joinedDF8.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#1028], [graduate_program#1009], Cross, BuildRight, false, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3389]
 +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]


+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 id| degree| department| school| id| name|graduate_program| spark_status|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 0|Masters|School of Informa...|UC Berkeley| 0| Bill Chambers| 0| [100]|
 1| Ph.D.| EECS|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 1| Ph.D.| EECS|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+

joinedDF8: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 6 more fields]

In [0]:
%scala
val joinedDF9 = graduateProgram.crossJoin(person)
joinedDF9.explain()
joinedDF9.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastNestedLoopJoin BuildRight, Cross, true
 :- LocalTableScan [id#1028, degree#1029, department#1030, school#1031]
 +- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=3456]
 +- LocalTableScan [id#1007, name#1008, graduate_program#1009, spark_status#1010]


+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 id| degree| department| school| id| name|graduate_program| spark_status|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+
 0|Masters|School of Informa...|UC Berkeley| 0| Bill Chambers| 0| [100]|
 0|Masters|School of Informa...|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
 0|Masters|School of Informa...|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 2|Masters| EECS|UC Berkeley| 0| Bill Chambers| 0| [100]|
 2|Masters| EECS|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
 2|Masters| EECS|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 1| Ph.D.| EECS|UC Berkeley| 0| Bill Chambers| 0| [100]|
 1| Ph.D.| EECS|UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
 1| Ph.D.| EECS|UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
+---+-------+--------------------+-----------+---+----------------+----------------+---------------+

joinedDF9: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 6 more fields]

Zadanie 2 


Użyj Generatora i stwórz dwie duże tabele po 1 milion wierszy i wykonaj dwa typy joinów, inner oraz left.  

Połącz tabele po tych samych kolumnach i użyj jednej metody z wykładów na usunięcie duplikatów.  

In [0]:
from pyspark.sql.functions import rand, expr

df_A = spark.range(1_000_000).withColumnRenamed("id", "id_A") \
    .withColumn("value_A", (rand() * 100).cast("int"))


df_B = spark.range(500_000, 1_500_000).withColumnRenamed("id", "id_B") \
    .withColumn("value_B", (rand() * 100).cast("int"))


In [0]:
inner_join_df = df_A.join(df_B, df_A.id_A == df_B.id_B, how="inner")
print("Liczba wierszy w INNER JOIN:", inner_join_df.count())


Liczba wierszy w INNER JOIN: 500000


In [0]:
left_join_df = df_A.join(df_B, df_A.id_A == df_B.id_B, how="left")
print("Liczba wierszy w LEFT JOIN:", left_join_df.count())


Liczba wierszy w LEFT JOIN: 1000000


In [0]:

display(inner_join_df.limit(10)) #mamy zduplikowaną kolumne id_B id_A


id_A,value_A,id_B,value_B
500000,29,500000,21
500001,25,500001,23
500002,62,500002,15
500003,31,500003,3
500004,31,500004,3
500005,53,500005,33
500006,43,500006,64
500007,53,500007,89
500008,69,500008,53
500009,56,500009,91


In [0]:
#Rozwiązanie 2 z wykładu
join_result_2 = inner_join_df.drop("id_B")

print("Liczba wierszy:", join_result_2.count())
display(join_result_2.limit(5))

Liczba wierszy: 500000


id_A,value_A,value_B
500000,29,21
500001,25,23
500002,62,15
500003,31,3
500004,31,3
